In [2]:
!pip install jq
# Install langchain and JSON loader from langchain
!pip install langchain
!pip install 'langchain[json_loader]'
!pip install -U langchain-community
import logging
!pip install streamlit
!pip install tiktoken
!pip install faiss-cpu
!pip install sentence-transformers
!pip install torch

In [3]:
from langchain.document_loaders import JSONLoader
from google.colab import drive
drive.mount('/content/drive')
loader=JSONLoader(file_path='/content/drive/MyDrive/newsarticle.json',text_content=False,jq_schema='.[].articleBody')
Newsarticles=loader.load()
Newsarticles[0].page_content # Should output the article body text


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'Sanjay Raut, a member of the Shiv Sena (UBT) party, responded to the Maharashtra chief minister\'s statement that Eknath Shinde "himself is Hamas" and that the Shiv Sena group led by Uddhav Thackeray is capable of collaborating with "Hamas and Lashkar-e-Taiba for their own selfishness" on Wednesday by claiming that Eknath Shinde is Hamas.\n\n\n\nRaut made fun of Shinde by claiming, "He himself is Hamas. Hamas and Lashkar-e-Taiba, two terrorist groups, are completely irrelevant in Maharashtra. But the BJP is to blame for sowing the worms in their (the Shinde faction\'s) thoughts, said Raut.\n\nWhen Shinde made a statement at the Tuesday Dussehra rally in Mumbai\'s Azad Maidan, Raut reacted to it. As part of the opposition alliance INDIA, Uddhav Thackeray\'s Shiv Sena (UBT) has formed an alliance with Congress and the Samajwadi Party. Shinde remarked of this alliance: "For their own selfishness, they will tie the knot with Hamas and Lashkar-e-Taiba."\n\nRaut highlighted that Shinde\'s a

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
r_splitter = RecursiveCharacterTextSplitter(separators=['\n\n', '\n', ' '], chunk_size=3000, chunk_overlap=300)

# Split the combined articles into chunks
chunks = r_splitter.split_documents(Newsarticles)

# Total number of chunks
total_chunks = len(chunks)
total_chunks

36984

In [5]:
from langchain.vectorstores import FAISS
import streamlit as st
import pickle

In [7]:
from sentence_transformers import SentenceTransformer

# Initialize SentenceTransformer model
encode = SentenceTransformer('all-mpnet-base-v2')

# Assuming 'chunks' is a list of 'Document' objects
# Use list comprehension to extract text content from each 'Document' object
texts = [chunk.page_content for chunk in chunks]

# Encode the text content using SentenceTransformer
vectors = encode.encode(texts)

# Check the shape of the encoded vectors
print(vectors.shape)

KeyboardInterrupt: 

In [ ]:
dim=vectors.shape[1]
import faiss
index=faiss.IndexFlatL2(dim)
index

In [ ]:
index.add(vectors)

In [ ]:
search_query='What happened at the Al-Shifa Hospital?'
vec=encode.encode(search_query)
vec.shape

In [ ]:
import numpy as np
svec=np.array(vec).reshape(1,-1)
svec.shape

In [ ]:
dist,indices=index.search(svec,k=3)

In [ ]:
flattened_indices = indices.flatten()

# Ensure that indices are integers
flattened_indices = [int(i) for i in flattened_indices]

# Retrieve relevant chunks
relevant_chunks = [chunks[i] for i in flattened_indices]

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Concatenate the relevant chunks' text
relevant_text = " ".join([chunk.page_content for chunk in relevant_chunks])

# Load a pre-trained model and tokenizer from Hugging Face (using a model suitable for QA, like T5 or BART)
model_name = "t5-large"  # You can change this to a different model if you prefer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Tokenize the input text
input_ids = tokenizer.encode(relevant_text, return_tensors="pt", max_length=1024, truncation=True)

# Generate the answer
output = model.generate(input_ids=input_ids, max_length=512, num_beams=4, early_stopping=True)

# Decode the generated text
answer = tokenizer.decode(output[0], skip_special_tokens=True)

# Print the generated answer
print("Generated Answer:", answer)